In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
import os
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt

class SUIMDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.img_dir = os.path.join(root_dir, 'images')
        self.mask_dir = os.path.join(root_dir, 'masks')

        self.images = sorted([f for f in os.listdir(self.img_dir) if f.endswith(('.jpg', '.png'))])
        self.masks = sorted([f for f in os.listdir(self.mask_dir) if f.endswith(('.jpg', '.png'))])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir, self.masks[idx])

        # Load Image and Mask
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        # CRITICAL: Resize both to match model input
        # Use NEAREST for masks to avoid creating fake class labels
        image = image.resize((256, 256), Image.BILINEAR)
        mask = mask.resize((256, 256), Image.NEAREST)

        if self.transform:
            image = self.transform(image)

        # Convert mask to long tensor and remap
        mask_tensor = torch.from_numpy(np.array(mask)).long()
        mask_tensor = remap_mask(mask_tensor)

        return image, mask_tensor

# Define transforms (Normalization is good practice for EfficientNet)
tfs = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset_path = os.path.join(path, "dataset")
full_ds = SUIMDataset(root_dir=dataset_path, transform=tfs)

# Split 80/20
train_size = int(0.8 * len(full_ds))
val_size = len(full_ds) - train_size
train_ds, val_ds = torch.utils.data.random_split(full_ds, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)

In [ ]:
# TO DO

import segmentation_models_pytorch as smp

# Initialize U-Net with EfficientNet-B1 backbone
model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=8  # 8 output classes for SUIM
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
# TO DO

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    return running_loss / len(loader)

def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for images, masks in loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            loss = criterion(outputs, masks)
            running_loss += loss.item()
    return running_loss / len(loader)

In [ ]:
# TO DO

import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# 1. Define Loss and Optimizer
# Using CrossEntropy as it's the standard for 8-class pixel classification
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# 2. Run Training
num_epochs = 5
train_losses = []
val_losses = []

print("Starting Training...")
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss = validate_one_epoch(model, val_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

# 3. Plot Loss Curves
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Training Loss', color='blue')
plt.plot(val_losses, label='Validation Loss', color='orange')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Multi-Class Segmentation Loss Curve')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# TO DO


model.eval()
with torch.no_grad():
    imgs, msks = next(iter(val_loader))
    preds = torch.argmax(model(imgs.to(device)), dim=1).cpu()

    for i in range(3): # Show first 3 samples
        plt.figure(figsize=(12, 4))
        plt.subplot(1, 3, 1); plt.imshow(imgs[i].permute(1,2,0)); plt.title("Input")
        plt.subplot(1, 3, 2); plt.imshow(msks[i]); plt.title("Ground Truth")
        plt.subplot(1, 3, 3); plt.imshow(preds[i]); plt.title("Prediction")
        plt.show()